In [0]:
ADLS_ACCOUNT        = "adbstoragev10"
SILVER_CONTAINER    = "silver"

ADLS_ROOT_SILVER    = f"abfss://{SILVER_CONTAINER}@{ADLS_ACCOUNT}.dfs.core.windows.net"
SILVER_NON_FOCUS_PATH = f"{ADLS_ROOT_SILVER}/non_focus"
SILVER_FOCUS_PATH = f"{ADLS_ROOT_SILVER}/focus"


CATALOG = "finops"

print("✅ Config done")
print(f"Silver path: {SILVER_NON_FOCUS_PATH}")
print(f"Silver path: {SILVER_FOCUS_PATH}")


In [0]:
df_non_focus = spark.table("finops.bronze.non_focus")

print(f"✅ Bronze non_focus loaded — {df_non_focus.count()} rows")

In [0]:
df_focus = spark.table("finops.bronze.focus")

print(f"✅ Bronze focus loaded — {df_focus.count()} rows")

In [0]:
from pyspark.sql.functions import from_json, to_json, get_json_object, col
from pyspark.sql.types import MapType, StringType

df_non_focus_silver = df_non_focus.selectExpr(
    "'NON-FOCUS' as RECORD_TYPE",
    "'ADB' as CLIENT",
    "'AZURE' as MEDIA_ID",
    "invoiceId as INVOICE_NUMBER",
    "CAST(NULL AS STRING) as INVOICE_DATE",
    "billingAccountId as PARENT_ACCOUNT_NUMBER",
    "SubscriptionId as ACCOUNT_NUMBER",
    "subscriptionName as ACCOUNT_NAME",
    "DATE_FORMAT(TO_DATE(date, 'MM/dd/yyyy'), 'yyyyMMdd') as USAGE_DATE",
    "'00:00:00' as USAGE_START_TIME",
    "CAST(quantity AS DECIMAL(20,10)) as QUANTITY",
    "unitOfMeasure as UNITS",
    "CASE WHEN costInBillingCurrency IS NULL THEN CAST(0.00 AS DECIMAL(20,10)) ELSE CAST(costInBillingCurrency AS DECIMAL(20,10)) END as VENDOR_COST",
    "CAST(0.00 AS DECIMAL(20,10)) as UPLIFT_AMOUNT",
    "CAST(0.00 AS DECIMAL(20,10)) as RERATED_COST",
    "CAST(0.00 AS DECIMAL(20,10)) as TAX_AMOUNT",
    "CASE WHEN costInBillingCurrency IS NULL THEN CAST(0.00 AS DECIMAL(20,10)) ELSE CAST(costInBillingCurrency AS DECIMAL(20,10)) END as TOTAL_CHARGES",
    "ProductName as CHARGE_DESCRIPTION",
    "meterCategory as CHARGE_CODE",
    "publisherType as OTHER_DESCRIPTION",
    "substring_index(ResourceId, '/', -1) as ASSET_ID",
    "'IAAS' as ASSET_TYPE",
    "serviceInfo2 as ASSET_STATUS",
    "CAST(NULL AS STRING) as COST_CENTER",
    "billingCurrency as CURRENCY_CD",
    "CAST(NULL AS STRING) as COUNTRY_CD",
    "consumedService as CATEGORY",
    "meterCategory as SUBCATEGORY",
    "substring(meterName, 1, 128) as NAME",
    "ProductName as PRODUCT",
    "meterSubCategory as SUPPLEMENTAL",
    "billingPeriodStartDate as BILLINGPERIODSTARTDATE",
    "billingPeriodEndDate as BILLINGPERIODENDDATE",
    "chargeType as RECORDTYPE",
    "additionalInfo as ADDITIONALINFO",
    "productOrderId as DEPARTMENTID",
    "productOrderName as DEPARTMENTNAME",
    "resourceLocation as REGION",
    "meterCategory as METERREGION",
    "resourceGroupName as RESOURCEGROUP",
    "ResourceId as INVENTORYID",
    "costCenter as ORIGIN_COST_CENTER",
    "tags as TAGS",
    "CAST(NULL AS STRING) as TBM_CHARGE_TYPE",
    "CAST(NULL AS STRING) as PROJECT_CODE",
    "CASE WHEN benefitId IS NOT NULL THEN benefitName ELSE '' END as BUSINESS_UNIT",
    "CAST(NULL AS STRING) as ENTERPRISE_LOCATION",
    "CAST(NULL AS STRING) as GL_EXPENSE_CODE",
    "pricingModel as MAP_CITY",
    "CAST(NULL AS STRING) as MAP_STATE",
    "term as MAP_COUNTRY",
    "CAST(NULL AS STRING) as MESSAGE_CODE",
    "'N' as GL_FLAG",
    "'N' as VENDORCOST_ADJ_FLAG",
    "get_json_object(to_json(from_json(additionalInfo, 'map<string,string>')), '$.ServiceType') as SERVER_TYPE",
    "CASE WHEN benefitId IS NOT NULL THEN benefitId ELSE '' END as RESERVATION_ID",
    "CAST(0.00 AS DECIMAL(20,10)) as INVOICE_COST",
    "CAST(NULL AS STRING) as SOURCE_ID",
    "CAST(NULL AS STRING) as RECORDNUMBER",
    "CAST(0.00 AS DECIMAL(20,10)) as RATE_DIFFERENCE",
    "CAST(0.00 AS DECIMAL(20,10)) as CONTRACT_DISCOUNT",
    "CAST(NULL AS STRING) as RATE_CHECK",
    "CAST(NULL AS STRING) as AVAILABILITY_ZONE",
    "billingAccountName as BILLING_ACCOUNT_NAME",
    "chargeType as CHARGE_CATEGORY",
    "CAST(NULL AS STRING) as CHARGE_CLASS",
    "frequency as CHARGE_FREQUENCY",
    "DATE_FORMAT(TO_DATE(date, 'MM/dd/yyyy'), 'yyyyMMdd') as CHARGE_PERIOD_END",
    """CASE WHEN benefitId IS NOT NULL THEN
    CASE WHEN UPPER(pricingModel) IN ('RESERVATION','SPOT') THEN 'USAGE'
    WHEN UPPER(pricingModel) = 'SAVINGSPLAN' THEN 'SPEND'
    ELSE '' END
    ELSE '' END as COMMITMENT_DISCOUNT_CATEGORY""",
    "benefitName as COMMITMENT_DISCOUNT_NAME",
    "CASE WHEN benefitId IS NOT NULL AND UPPER(chargeType) = 'USAGE' THEN 'USED' ELSE '' END as COMMITMENT_DISCOUNT_STATUS",
    "CASE WHEN benefitId IS NOT NULL THEN pricingModel ELSE '' END as COMMITMENT_DISCOUNT_TYPE",
    "CAST((unitPrice * quantity) AS DECIMAL(20,10)) as CONTRACTED_COST",
    "CAST(unitPrice AS DECIMAL(20,10)) as CONTRACTED_UNIT_PRICE",
    "CAST(0.00 AS DECIMAL(20,10)) as EFFECTIVE_COST",
    "CAST(NULL AS STRING) as INVOICE_ISSUER",
    "CAST(paygCostInBillingCurrency AS DECIMAL(20,10)) as LIST_COST",
    """CAST(CASE WHEN UPPER(chargeType) = 'USAGE' AND UPPER(pricingModel) IN ('SAVINGSPLAN','RESERVATION')
    THEN unitPrice ELSE PayGPrice END AS DECIMAL(20,10)) as LIST_UNIT_PRICE""",
    "pricingModel as PRICING_CATEGORY",
    "CAST(quantity AS DECIMAL(20,10)) as PRICING_QUANTITY",
    "unitOfMeasure as PRICING_UNIT",
    "provider as PROVIDER",
    "publisherName as PUBLISHER",
    "meterRegion as REGION_NAME",
    "CAST(NULL AS STRING) as RESOURCE_NAME",
    "CAST(NULL AS STRING) as RESOURCE_TYPE",
    "CAST(NULL AS STRING) as SERVICE_SUBCATEGORY",
    "ProductId as SKU_ID",
    "CAST(NULL AS STRING) as SKU_PRICE_DETAILS",
    "ProductId as SKU_PRICE_ID",
    "CAST(NULL AS STRING) as CAPACITY_RESERVATION_ID",
    "CAST(NULL AS STRING) as CAPACITY_RESERVATION_STATUS",
    "CAST(CASE WHEN benefitId IS NOT NULL THEN quantity ELSE 0 END AS DECIMAL(20,10)) as OMMITMENT_DISCOUNT_QUANTITY",
    "get_json_object(to_json(from_json(additionalInfo, 'map<string,string>')), '$.RINormalizationRatio') as COMMITMENT_DISCOUNT_UNIT",
    "meterRegion as SKU_METER",
    "meterId as METER_ID",
    "CAST(effectivePrice AS DECIMAL(20,10)) as UNIT_PRICE",
    "CAST(NULL AS STRING) as PURCHASE_OPTION"
)

print(f"✅ Transformation done — {df_non_focus_silver.count()} rows")

In [0]:
df_focus_silver = df_focus.selectExpr(
    "'FOCUS' as RECORD_TYPE",
    "'ADB' as CLIENT",
    "'AZURE' as MEDIA_ID",
    "x_InvoiceId as INVOICE_NUMBER",
    "CAST(NULL AS STRING) as INVOICE_DATE",
    "split(split(BillingAccountId, 'billingAccounts/')[1], '/billingProfiles')[0] as PARENT_ACCOUNT_NUMBER",
    "SubAccountId as ACCOUNT_NUMBER",
    "SubAccountName as ACCOUNT_NAME",
    "replace(substring(ChargePeriodStart, 1, 10), '-', '') as USAGE_DATE",
    "CASE WHEN TRIM(substring(ChargePeriodStart, 12, 8)) = '' THEN '00:00:00' ELSE substring(ChargePeriodStart, 12, 8) END as USAGE_START_TIME",
    "NVL(CAST(ConsumedQuantity AS DECIMAL(20,10)), 0) as QUANTITY",
    "ConsumedUnit as UNITS",
    "CAST(BilledCost AS DECIMAL(20,10)) as VENDOR_COST",
    "CAST(0.00 AS DECIMAL(20,10)) as UPLIFT_AMOUNT",
    "CAST(0.00 AS DECIMAL(20,10)) as RERATED_COST",
    "CAST(0.00 AS DECIMAL(20,10)) as TAX_AMOUNT",
    "CAST(BilledCost AS DECIMAL(20,10)) as TOTAL_CHARGES",
    "ChargeDescription as CHARGE_DESCRIPTION",
    "x_SkuMeterCategory as CHARGE_CODE",
    "x_PublisherCategory as OTHER_DESCRIPTION",
    "substring_index(ResourceId, '/', -1) as ASSET_ID",
    "'IAAS' as ASSET_TYPE",
    "CAST(NULL AS STRING) as ASSET_STATUS",
    "x_CostCenter as COST_CENTER",
    "BillingCurrency as CURRENCY_CD",
    "CAST(NULL AS STRING) as COUNTRY_CD",
    "ServiceName as CATEGORY",
    "x_SkuMeterCategory as SUBCATEGORY",
    "substring(x_SkuMeterName, 1, 128) as NAME",
    "x_SkuMeterSubcategory as PRODUCT",
    "CAST(NULL AS STRING) as SUPPLEMENTAL",
    "replace(substring(BillingPeriodStart, 1, 10), '-', '') as BILLINGPERIODSTARTDATE",
    "replace(substring(BillingPeriodEnd, 1, 10), '-', '') as BILLINGPERIODENDDATE",
    "ChargeCategory as RECORDTYPE",
    "x_SkuDetails as ADDITIONALINFO",
    "x_SkuOrderId as DEPARTMENTID",
    "x_SkuOrderName as DEPARTMENTNAME",
    "RegionId as REGION",
    "CAST(NULL AS STRING) as METERREGION",
    "x_ResourceGroupName as RESOURCEGROUP",
    "ResourceId as INVENTORYID",
    "x_CostCenter as ORIGIN_COST_CENTER",
    "Tags as TAGS",
    "ServiceCategory as TBM_CHARGE_TYPE",
    "CAST(NULL AS STRING) as PROJECT_CODE",
    "CommitmentDiscountName as BUSINESS_UNIT",
    "CAST(NULL AS STRING) as ENTERPRISE_LOCATION",
    "CAST(NULL AS STRING) as GL_EXPENSE_CODE",
    "CAST(NULL AS STRING) as MAP_CITY",
    "x_SkuOfferId as MAP_STATE",
    "x_SkuTerm as MAP_COUNTRY",
    "CAST(NULL AS STRING) as MESSAGE_CODE",
    "'N' as GL_FLAG",
    "'N' as VENDORCOST_ADJ_FLAG",
    "substring(x_SkuDetails, 1, 39) as SERVER_TYPE",
    "CommitmentDiscountId as RESERVATION_ID",
    "CAST(0.00 AS DECIMAL(20,10)) as INVOICE_COST",
    "CAST(NULL AS STRING) as SOURCE_ID",
    "CAST(NULL AS STRING) as RECORDNUMBER",
    "CAST(0.00 AS DECIMAL(20,10)) as RATE_DIFFERENCE",
    "CAST(0.00 AS DECIMAL(20,10)) as CONTRACT_DISCOUNT",
    "CAST(NULL AS STRING) as RATE_CHECK",
    "CAST(NULL AS STRING) as AVAILABILITY_ZONE",
    "BillingAccountName as BILLING_ACCOUNT_NAME",
    "ChargeCategory as CHARGE_CATEGORY",
    "ChargeClass as CHARGE_CLASS",
    "ChargeFrequency as CHARGE_FREQUENCY",
    "replace(substring(ChargePeriodEnd, 1, 10), '-', '') as CHARGE_PERIOD_END",
    "CommitmentDiscountCategory as COMMITMENT_DISCOUNT_CATEGORY",
    "CommitmentDiscountName as COMMITMENT_DISCOUNT_NAME",
    "CommitmentDiscountStatus as COMMITMENT_DISCOUNT_STATUS",
    "CommitmentDiscountType as COMMITMENT_DISCOUNT_TYPE",
    "CAST(ContractedCost AS DECIMAL(20,10)) as CONTRACTED_COST",
    "CAST(ContractedUnitPrice AS DECIMAL(20,10)) as CONTRACTED_UNIT_PRICE",
    "CAST(EffectiveCost AS DECIMAL(20,10)) as EFFECTIVE_COST",
    "InvoiceIssuerName as INVOICE_ISSUER",
    "CAST(ListCost AS DECIMAL(20,10)) as LIST_COST",
    "CAST(ListUnitPrice AS DECIMAL(20,10)) as LIST_UNIT_PRICE",
    "PricingCategory as PRICING_CATEGORY",
    "CAST(PricingQuantity AS DECIMAL(20,10)) as PRICING_QUANTITY",
    "PricingUnit as PRICING_UNIT",
    "ProviderName as PROVIDER",
    "PublisherName as PUBLISHER",
    "RegionName as REGION_NAME",
    "ResourceName as RESOURCE_NAME",
    "ResourceType as RESOURCE_TYPE",
    "CAST(NULL AS STRING) as SERVICE_SUBCATEGORY",
    "SkuId as SKU_ID",
    "CAST(NULL AS STRING) as SKU_PRICE_DETAILS",
    "SkuPriceId as SKU_PRICE_ID",
    "CAST(NULL AS STRING) as CAPACITY_RESERVATION_ID",
    "CAST(NULL AS STRING) as CAPACITY_RESERVATION_STATUS",
    "CAST(0.00 AS DECIMAL(20,10)) as COMMITMENT_DISCOUNT_QUANTITY",
    "CAST(NULL AS STRING) as COMMITMENT_DISCOUNT_UNIT",
    "x_SkuRegion as SKU_METER",
    "x_SkuMeterId as METER_ID",
    "CAST(x_EffectiveUnitPrice AS DECIMAL(20,10)) as UNIT_PRICE",
    "CAST(NULL AS STRING) as PURCHASE_OPTION"
)

print(f"✅ FOCUS Silver transformation done — {df_focus_silver.count()} rows")

In [0]:
df_non_focus_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://silver@adbstoragev10.dfs.core.windows.net/non_focus")

print("✅ Non-FOCUS Silver written")

In [0]:
df_focus_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://silver@adbstoragev10.dfs.core.windows.net/focus")

print("✅ FOCUS Silver written")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS finops.silver.focus
LOCATION 'abfss://silver@adbstoragev10.dfs.core.windows.net/focus';

CREATE TABLE IF NOT EXISTS finops.silver.non_focus
LOCATION 'abfss://silver@adbstoragev10.dfs.core.windows.net/non_focus';

In [0]:
%sql
SELECT * from finops.silver.non_focus

In [0]:
%sql
SELECT * from finops.silver.focus;
